# Chandra OCR 2 — Test Notebook
**Yêu cầu:** Runtime → Change runtime type → **T4 GPU** (Colab Pro)

Notebook này chỉ test **Chandra OCR 2** (không có Marker).
Chandra OCR 2 dùng model Qwen3.5 → cần **transformers 5.x** → KHÔNG pin `<5.0.0`.

In [ ]:
# ── 0. Kiểm tra GPU ──────────────────────────────────────────────────────────
import subprocess
r = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,memory.free',
                    '--format=csv,noheader'], capture_output=True, text=True)
print('GPU:', r.stdout.strip())

In [ ]:
%%time
# ── 1. Cài dependencies ───────────────────────────────────────────────────────
# KHÔNG pin transformers — Chandra OCR 2 dùng qwen3_5 cần transformers 5.x

!pip install -q 'chandra-ocr[hf]' bitsandbytes pymupdf
print('✅ Cài xong.')

In [ ]:
# ── 2. Upload file (ảnh hoặc PDF) ────────────────────────────────────────────
from google.colab import files
import io, os
from PIL import Image
import fitz  # PyMuPDF

uploaded = files.upload()
fname = list(uploaded.keys())[0]
fdata = uploaded[fname]
ext = os.path.splitext(fname)[1].lower()

is_pdf = (ext == '.pdf')

if is_pdf:
    pdf_path = '/content/test_doc.pdf'
    with open(pdf_path, 'wb') as f:
        f.write(fdata)
    doc = fitz.open(pdf_path)
    pil_pages = []
    for page in doc:
        pix = page.get_pixmap(matrix=fitz.Matrix(2, 2))  # ~144 DPI
        pil_pages.append(Image.frombytes('RGB', [pix.w, pix.h], pix.samples))
    doc.close()
    pil_image = pil_pages[0]
    print(f'✅ PDF: {fname}  →  {len(pil_pages)} trang')
    display(pil_image)
else:
    pil_image = Image.open(io.BytesIO(fdata)).convert('RGB')
    pil_pages = [pil_image]
    print(f'✅ Ảnh: {fname}  →  {pil_image.size}')
    display(pil_image)

In [ ]:
# ── 3. Load Chandra model ─────────────────────────────────────────────────────
import os, time, torch, gc
os.environ['TORCH_DEVICE'] = 'cuda'

free_mb = torch.cuda.mem_get_info()[0] // (1024 ** 2)
use_4bit = free_mb < 10000
mode_label = '4-bit NF4' if use_4bit else 'fp16'
print(f'VRAM trống: {free_mb}MB → {mode_label}')

from chandra.model import InferenceManager

if use_4bit:
    from transformers import AutoModelForImageTextToText, AutoProcessor, BitsAndBytesConfig
    import chandra.model as _cm
    from chandra.model import settings as _cs

    bnb = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type='nf4',
        llm_int8_enable_fp32_cpu_offload=True,
    )

    try:
        from bitsandbytes.nn.modules import Params4bit
        _orig_new = Params4bit.__new__
        def _patched_new(cls, *args, **kwargs):
            kwargs.pop('_is_hf_initialized', None)
            return _orig_new(cls, *args, **kwargs)
        Params4bit.__new__ = staticmethod(_patched_new)
    except Exception:
        pass

    def _load_4bit():
        model = AutoModelForImageTextToText.from_pretrained(
            _cs.MODEL_CHECKPOINT,
            quantization_config=bnb,
            device_map='auto',
            max_memory={0: f'{free_mb - 500}MiB', 'cpu': '48GiB'},
            trust_remote_code=True,
        ).eval()
        processor = AutoProcessor.from_pretrained(_cs.MODEL_CHECKPOINT)
        processor.tokenizer.padding_side = 'left'
        model.processor = processor
        return model

    _orig_load = _cm.load_model
    _cm.load_model = _load_4bit
    print(f'Loading Chandra {mode_label}...')
    _t_load_start = time.perf_counter()
    manager = InferenceManager(method='hf')
    _cm.load_model = _orig_load
else:
    print(f'Loading Chandra {mode_label}...')
    _t_load_start = time.perf_counter()
    manager = InferenceManager(method='hf')

t_load = time.perf_counter() - _t_load_start
print(f'✅ Load xong: {t_load:.1f}s')

In [ ]:
# ── 4. Chạy OCR ──────────────────────────────────────────────────────────────
import base64, re
from io import BytesIO
from chandra.model.schema import BatchInputItem

def _embed_images(md_text, images_dict):
    """Thay ![](<filename>) bằng base64 data URI để hiển thị trong Colab."""
    if not images_dict:
        return md_text
    def _replace(m):
        alt, src = m.group(1), m.group(2)
        img = images_dict.get(src)
        if img is None:
            return m.group(0)
        buf = BytesIO()
        img.save(buf, format='PNG')
        b64 = base64.b64encode(buf.getvalue()).decode()
        return f'![{alt}](data:image/png;base64,{b64})'
    return re.sub(r'!\[([^\]]*)\]\(([^)]+)\)', _replace, md_text)

page_times = []
all_md = []

for i, page_img in enumerate(pil_pages):
    label = f'Trang {i+1}/{len(pil_pages)}' if len(pil_pages) > 1 else 'Ảnh'
    print(f'{label}...', end=' ', flush=True)
    t0 = time.perf_counter()

    batch = [BatchInputItem(image=page_img, prompt_type='ocr_layout')]
    results = manager.generate(batch)
    result = results[0]

    page_md = getattr(result, 'markdown', '') or ''
    images  = getattr(result, 'images',   {}) or {}

    # Embed ảnh figures/charts thành base64 để Markdown hiển thị đúng
    if images:
        page_md = _embed_images(page_md, images)
        print(f'(+{len(images)} ảnh)', end=' ')

    elapsed = time.perf_counter() - t0
    page_times.append(elapsed)
    all_md.append(page_md)
    print(f'{elapsed:.1f}s  ({len(page_md)} ký tự)')

chandra_md = '\n\n---\n\n'.join(all_md).strip()
t_ocr = sum(page_times)

# ── Timing summary ────────────────────────────────────────────────────────────
print()
print('╔══════════════════════════════════════╗')
print('║          TIMING SUMMARY              ║')
print('╠══════════════════════════════════════╣')
print(f'║  Mode        : {mode_label:<22}║')
print(f'║  Model load  : {t_load:>7.1f}s                 ║')
print(f'║  OCR total   : {t_ocr:>7.1f}s  ({len(pil_pages)} trang)       ║')
if len(pil_pages) > 1:
    avg = t_ocr / len(pil_pages)
    print(f'║  Avg/trang   : {avg:>7.1f}s                 ║')
    tpm = len(pil_pages) / (t_ocr / 60)
    print(f'║  Throughput  : {tpm:>7.1f} trang/phút      ║')
print(f'║  Output      : {len(chandra_md):>7,} ký tự             ║')
print('╚══════════════════════════════════════╝')

if len(pil_pages) > 1:
    print('\nChi tiết từng trang:')
    for i, t in enumerate(page_times):
        bar = '█' * int(t / max(page_times) * 20)
        print(f'  Trang {i+1:2d}: {t:5.1f}s  {bar}')

In [ ]:
# ── 5. Hiển thị kết quả dạng Markdown đẹp ────────────────────────────────────
from IPython.display import Markdown
Markdown(chandra_md)

In [ ]:
# ── 6. (Tuỳ chọn) Lưu kết quả ra file ───────────────────────────────────────
out_path = '/content/chandra_result.md'
with open(out_path, 'w', encoding='utf-8') as f:
    f.write(chandra_md)
print(f'✅ Đã lưu: {out_path}')
files.download(out_path)